# Timing on 120 pairs, one protocol for every method (`tab:runtime`)

Cell 3 (TF1, Texture Mixer) and cell 6 (CNT) time every stage: encode, decode at 256px and 128px, OT transport/solve, hard-assignment solve, Texture Mixer encode/decode/interp.

**Needs:** DTD, the CNT checkpoint, the Texture Mixer checkpoint, one GPU with nothing else running (wall-clock timings are hardware-specific and not meaningfully reproducible on different hardware).

**Maintained runnable path:** `experiments/runtime_120.py` reproduces the CNT/TM columns of this timing protocol (not Vacher's or GPT's, which come from separate protocols -- see docs/INVENTORY.md section 0 item 7). Absolute numbers will differ from the paper's A100 run on any other GPU; this only lets you re-measure on your own hardware.

# Timing on the 120 pairs — one protocol for every method

Two phases because Texture Mixer is TF1 and CNT is torch; they cannot share a kernel.

**Phase 1 (cells 1–3, TensorFlow):** Texture Mixer setup verbatim from `tm_scale1000_headtohead`, then its timing on the 120 pairs. Then **Runtime → Restart session**.

**Phase 2 (cells 4–6, torch):** CNT setup, our methods' timing on the same pairs, and a final cell that merges both CSVs into one table and prints the `\newcommand` lines.

Protocol for both phases: all image loading and resizing *before* the clock; 3 warm-up pairs; encode timed separately from the per-η step; median over the 120 pairs with IQR; `midpoint = enc + method + one decode`, `path11 = enc + method + 11 decodes`. Torch stages are wrapped in `cuda.synchronize()`; TF1 `sess.run` is synchronous. No disk writes inside any timed region — the old 0.375 s figure included a PNG write to Drive.

Side note this notebook makes explicit: `load128` in the original TM cell 3 resized every DTD image to 128×128 *before* saving `tm_scale1000/A|B`. That is why every endpoint in the pair set is 128 px and why all CNT-based methods have been running on 2× upsampled inputs.


In [ ]:
# ============================================================
# PHASE 1 (TensorFlow), CELL 1 — Texture Mixer setup, verbatim from tm_scale1000_headtohead. No torch in this phase.
# ============================================================
import os, sys, types, importlib, importlib.util, importlib.machinery, glob
import numpy as np
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
ImageFile.LOAD_TRUNCATED_IMAGES = True

from google.colab import drive
drive.mount('/content/drive')
BASE = os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
TM_STORE = f'{BASE}/texturemixer'
TM_OUT = f'{TM_STORE}/midpoints'
os.makedirs(TM_OUT, exist_ok=True)

# --- repo ---
os.chdir('/content')
if not os.path.isdir('/content/TextureMixer'):
    !git clone -q https://github.com/ningyu1991/TextureMixer.git
if not os.path.isdir('/content/TextureMixer/tensorflow_vgg'):
    !git clone -q https://github.com/machrisaa/tensorflow-vgg.git /content/TextureMixer/tensorflow_vgg
os.chdir('/content/TextureMixer'); sys.path.insert(0, '/content/TextureMixer')

# --- Python 3.12 removed `imp` ---
if 'imp' not in sys.modules:
    _imp = types.ModuleType('imp')
    def _load_source(name, pathname, file=None):
        spec = importlib.util.spec_from_file_location(name, pathname)
        mod = importlib.util.module_from_spec(spec); sys.modules[name] = mod
        spec.loader.exec_module(mod); return mod
    def _find_module(name, path=None):
        spec = importlib.machinery.PathFinder().find_spec(name, path)
        if spec is None: raise ImportError(name)
        return (None, spec.origin, ('.py','r',1))
    _imp.load_source=_load_source; _imp.find_module=_find_module
    _imp.new_module=lambda n: types.ModuleType(n); _imp.reload=importlib.reload
    _imp.acquire_lock=lambda: None; _imp.release_lock=lambda: None; _imp.PY_SOURCE=1
    sys.modules['imp'] = _imp
    print('imp shim installed')

# --- TF1 graph mode on TF2 ---
import tensorflow as _tf2
print('TF', _tf2.__version__, '| GPU:', _tf2.config.list_physical_devices('GPU'))
tf1 = _tf2.compat.v1
tf1.disable_eager_execution(); tf1.disable_v2_behavior()
tf1.logging.set_verbosity(tf1.logging.ERROR)
if not hasattr(tf1, 'Dimension'): tf1.Dimension = getattr(_tf2, 'Dimension', int)

!pip install -q tf_slim
import tf_slim as _slim
_contrib = types.ModuleType('tensorflow.contrib')
_contrib.slim=_slim; _contrib.framework=_slim; _contrib.layers=_slim
sys.modules['tensorflow.contrib']=_contrib
sys.modules['tensorflow.contrib.slim']=_slim
sys.modules['tensorflow.contrib.framework']=_slim
sys.modules['tensorflow.contrib.layers']=_slim
tf1.contrib=_contrib
sys.modules['tensorflow'] = tf1                 # repo does `import tensorflow as tf`
print('tf_slim wired into tf.contrib')

# --- Python-3 syntax fixes in the repo ---
_p='/content/TextureMixer/tfutil.py'; _s=open(_p).read()
_s=_s.replace('v.shape.ndims is 0','v.shape.ndims == 0').replace('v.shape.ndims is 1','v.shape.ndims == 1')
open(_p,'w').write(_s)

# --- checkpoint (cached on Drive) ---
CKPT = f'{TM_STORE}/earth_texture/network-final.pkl'
if not os.path.exists(CKPT):
    !pip install -q gdown
    !gdown -q 1ObAFBPGaRJFo11LUa0qNhRX14nTEWKC1 -O /content/earth.zip
    !cd /content && unzip -oq earth.zip -d "{TM_STORE}"
print('checkpoint present:', os.path.exists(CKPT))

# --- load networks (9-tuple, bind by name) ---
import tfutil, misc
sess = tfutil.create_session(force_as_default=True)
obj = misc.load_pkl(CKPT)
nets = {getattr(o,'name',f'_{i}'): o for i,o in enumerate(obj)}
print('components:', list(nets))
Es_zg, Es_zl, Gs = nets['Es_zg'], nets['Es_zl'], nets['Gs']
print('LOADED OK — Gs', Gs.input_shapes, '->', Gs.output_shapes)

In [ ]:
# ============================================================
# PHASE 1, CELL 2 — Texture Mixer encode/generate, verbatim (sanity plot kept)
# ============================================================
import random
TM_RES = 128
LATENT_IDX = 0          # 0 = mean (verified); 1 = second moment

def tm_prep(pil):
    a = np.asarray(pil.convert('RGB').resize((TM_RES,)*2), dtype=np.float32)
    return (a/127.5 - 1.0).transpose(2,0,1)[None]

def tm_to_pil(arr):
    return Image.fromarray(np.clip((arr[0].transpose(1,2,0)+1.0)*127.5,0,255).astype(np.uint8))

def tm_encode(pil):
    x = tm_prep(pil)
    return (Es_zg.run(x, return_as_list=True)[LATENT_IDX],
            Es_zl.run(x, return_as_list=True)[LATENT_IDX])

def tm_generate(zg, zl):
    zg_t = np.tile(zg, (1,1,zl.shape[2],zl.shape[3]))
    return tm_to_pil(Gs.run(zg_t, zl, return_as_list=True)[0])

def texturemixer_interp(A, B, weight=0.5):
    zgA, zlA = tm_encode(A); zgB, zlB = tm_encode(B)
    return tm_generate((1-weight)*zgA + weight*zgB, (1-weight)*zlA + weight*zlB)

# --- DTD (a few reads, with retry for Drive EIO) ---
DTD_ROOT = f'{BASE}/dtd/images'
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(c,s): return random.Random(s).choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
def safe_open(p, tries=4):
    import time as _t
    for i in range(tries):
        try:
            im = Image.open(p); im.load(); return im.convert('RGB')
        except OSError:
            if i == tries-1: raise
            _t.sleep(0.5*(i+1))
print(f'{len(cats)} categories')

# --- reconstruction sanity check ---
probe = safe_open(dtd_path(cats[20], 7))
zg, zl = tm_encode(probe)
fig, ax = plt.subplots(1,2, figsize=(6,3.2))
ax[0].imshow(probe.resize((TM_RES,)*2)); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(tm_generate(zg,zl)); ax[1].set_title('Gs(E(input))'); ax[1].axis('off')
plt.tight_layout(); plt.show()
print('>>> if the right panel resembles the left, continue to cell 3')

In [ ]:
# ============================================================
# PHASE 1, CELL 3 — Texture Mixer timing on the 120 pairs (paper protocol). Reads the 128px A/B PNGs the pair set already stores.
# ============================================================
import time, pandas as pd, numpy as np
OUT_ROOT=f'{BASE}/tm_scale1000'; KS120=sorted(int(k) for k in pd.read_csv(f'{BASE}/pairs_120/pairs.csv').k); print(len(KS120),'pairs')
def load_pair(k): return Image.open(f'{OUT_ROOT}/A/{k:04d}.png').convert('RGB'), Image.open(f'{OUT_ROOT}/B/{k:04d}.png').convert('RGB')
IMGS={k:load_pair(k) for k in KS120}                                    # all I/O before the clock
XS={k:(tm_prep(a),tm_prep(b)) for k,(a,b) in IMGS.items()}             # resize + normalise before the clock too (mirrors our untimed cnt_load)
def tm_enc_arr(x): return Es_zg.run(x,return_as_list=True)[LATENT_IDX], Es_zl.run(x,return_as_list=True)[LATENT_IDX]
def tm_gen_arr(zg,zl): zg_t=np.tile(zg,(1,1,zl.shape[2],zl.shape[3])); return Gs.run(zg_t,zl,return_as_list=True)[0]   # generator only; PIL conversion excluded, as tanh->PIL is excluded on our side
for k in KS120[:3]:                                                    # warm-up
    a,b=XS[k]; (zgA,zlA),(zgB,zlB)=tm_enc_arr(a),tm_enc_arr(b); tm_gen_arr(0.5*(zgA+zgB),0.5*(zlA+zlB))
rows=[]
for k in KS120:
    a,b=XS[k]
    t=time.perf_counter(); (zgA,zlA),(zgB,zlB)=tm_enc_arr(a),tm_enc_arr(b); t_enc=time.perf_counter()-t
    t=time.perf_counter(); _=tm_gen_arr(0.5*(zgA+zgB),0.5*(zlA+zlB)); t_gen=time.perf_counter()-t
    rows.append(dict(k=k,enc=t_enc,gen=t_gen,midpoint=t_enc+t_gen,path11=t_enc+11*t_gen))
T=pd.DataFrame(rows); T.to_csv(f'{BASE}/timing_120_tm.csv',index=False)
med,q1,q3=T.median(numeric_only=True),T.quantile(0.25,numeric_only=True),T.quantile(0.75,numeric_only=True)
print('Texture Mixer, n=%d, median [IQR] seconds'%len(T))
for c_ in ['enc','gen','midpoint','path11']: print(f'  {c_:9} {med[c_]:.3f}  [{q1[c_]:.3f}, {q3[c_]:.3f}]')
print('\n% numbers.tex:'); print(f'\\newcommand{{\\TMsec}}{{{med.midpoint:.2f}}}\\newcommand{{\\TMInterpSec}}{{{med.gen:.3f}}}\\newcommand{{\\PathSecTM}}{{{med.path11:.2f}}}')
print('\n>>> PHASE 1 done. Runtime -> Restart session, then run cells 4-6.')

## Restart the runtime here (Runtime → Restart session). Then continue with the torch phase.


In [ ]:
# ============================================================
# PHASE 2 (torch), CELL 4 — SETUP — CNT + our interpolators (from tm_headtohead cell 0, observers dropped; CNT pinned to 0726930)
# ============================================================
!pip install -q einops
import os, sys, glob, random, time, gc, math
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
from scipy.stats import wilcoxon
from scipy.linalg import sqrtm
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES = True
device = 'cuda' if torch.cuda.is_available() else 'cpu'

BASE = os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
TM_OUT = f'{BASE}/texturemixer/midpoints'
OUT = f'{BASE}/baselines_texturemixer'; os.makedirs(OUT, exist_ok=True)
TMP = '/content/tmp_imgs'; os.makedirs(TMP, exist_ok=True)

# --- DTD on local disk (Drive FUSE throws EIO under bursty reads) ---
LOCAL_DTD = '/content/dtd_images'
if len(glob.glob(f'{LOCAL_DTD}/*/*.jpg')) < 5000:
    os.system(f'cp -r "{BASE}/dtd/images" {LOCAL_DTD}')
DTD_ROOT = LOCAL_DTD
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(c,s): return random.Random(s).choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
print(f'{len(cats)} categories, {len(glob.glob(f"{DTD_ROOT}/*/*.jpg"))} images')

# --- CNT ---
os.chdir('/content'); os.system('rm -rf cnt-siga24')
os.system('git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24 && cd cnt-siga24 && git checkout -q 07269302610c92ce38e2dbbe31c492fd09e1c73e')
os.system('pip install -q -r cnt-siga24/requirements.in')
os.system('pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT_DIR='/content/cnt-siga24'; os.chdir(CNT_DIR); sys.path.insert(0,CNT_DIR)
from src.inference.core import TexAutoEncoderInference, load_image
model = TexAutoEncoderInference(ckpt_path=Path(f"{CNT_DIR}/ckpts/finetuned-model256.ckpt"), device=device)
IMG_SIZE = 256
FMS = [round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]

_tc=[0]
def cnt_load(x, size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x), size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp)
    return load_image(Path(fp), size).to(device)
def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy()
    return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob, L["layer_level_layouts"][-1][0]["blob_features"].clone(), \
                     L["layer_level_layouts"][-1][0]["score_img"].clone()
def decode_interp(blob_ref, bf, si):
    with torch.no_grad():
        L=model.splat(blob_ref)
        L["layer_level_layouts"][-1][0]["blob_features"]=bf
        L["layer_level_layouts"][-1][0]["score_img"]=si
        return model.decode(L)
def build_style_maps(bf,si,fms):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms):
        s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref, sm):
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms):
            s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1)
        x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def sinkhorn_transport(src,tgt,eps=0.05,n_iters=100):
    N,M=src.shape[0],tgt.shape[0]; C=torch.cdist(src,tgt).pow(2); K=torch.exp(-C/eps)
    a=torch.ones(N,1,device=src.device)/N; b=torch.ones(M,1,device=src.device)/M; u=torch.ones(N,1,device=src.device)
    for _ in range(n_iters): v=b/(K.T@u+1e-10); u=a/(K@v+1e-10)
    P=u*K*v.T; return (P/(P.sum(1,keepdim=True)+1e-10))@tgt
def build_d(ia, ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def interp_ot(d, eta, n_sub=4096):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); src,tgt=sa[L][0],sb[L][0]; C,H,W=src.shape
    sf=src.permute(1,2,0).reshape(-1,C); tf=tgt.permute(1,2,0).reshape(-1,C); N=sf.shape[0]
    i_s=torch.randperm(N,device=device)[:n_sub]; i_t=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad(): tt=sinkhorn_transport(sf[i_s],tf[i_t])
    nn=torch.cdist(sf,sf[i_s]).argmin(1)
    it=((1-eta)*sf+eta*tt[nn]).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    return decode_with_style_maps(d["blob_a"], {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)})
def interp_ot_sym(d, eta, n_sub=4096):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sa[L][0],sb[L][0]; C,H,W=A.shape
    af=A.permute(1,2,0).reshape(-1,C); bf=B.permute(1,2,0).reshape(-1,C); N=af.shape[0]
    ia=torch.randperm(N,device=device)[:n_sub]; ib=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad():
        tab=sinkhorn_transport(af[ia],bf[ib]); tba=sinkhorn_transport(bf[ib],af[ia])
    na=torch.cdist(af,af[ia]).argmin(1); nb=torch.cdist(bf,bf[ib]).argmin(1)
    it=(0.5*(((1-eta)*af+eta*tab[na])+(eta*bf+(1-eta)*tba[nb]))).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    return decode_with_style_maps(d["blob_a"], {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)})
def interp_lin_pixel(d, eta):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"], {s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def interp_lin_texton(d, eta):
    return decode_interp(d["blob_a"], (1-eta)*d["bf_a"]+eta*d["bf_b"], (1-eta)*d["si_a"]+eta*d["si_b"])
FNS = {'pixel_linear':interp_lin_pixel,'texton_linear':interp_lin_texton,
       'ot':interp_ot,'ot_sym':interp_ot_sym}


from scipy.optimize import linear_sum_assignment
print('setup ready:',torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# PHASE 2, CELL 5 — OUR SIDE — 120 pairs, per-stage wall clock, medians (protocol of numbers.tex "Timing" block)
# ============================================================
import json; json.dump({n:float(v) for n,v in MAC.items()},open(f'{BASE}/timing_120_macros.json','w'),indent=1)
SCALE=f'{BASE}/tm_scale1000'; KS120=sorted(int(k) for k in pd.read_csv(f'{BASE}/pairs_120/pairs.csv').k); print(len(KS120),'pairs')
def pair_imgs(k): return Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'), Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')
def sync(): torch.cuda.synchronize()
def tic(): sync(); return time.perf_counter()
NSUB=4096; NEXACT=1024

def stages(k, seed=0):
    A,B=pair_imgs(k); ia,ib=cnt_load(A),cnt_load(B)                  # image load/resize: not timed (I/O, identical for every method)
    t=tic(); ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib); t_enc=tic()-t
    d={"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
    sa=build_style_maps(bfa,sia,FMS); sb=build_style_maps(bfb,sib,FMS); L=max(FMS); src,tgt=sa[L][0],sb[L][0]; C,H,W=src.shape
    sf=src.permute(1,2,0).reshape(-1,C); tf=tgt.permute(1,2,0).reshape(-1,C); N=sf.shape[0]
    g=torch.Generator(device=device).manual_seed(seed); i_s=torch.randperm(N,generator=g,device=device)[:NSUB]; i_t=torch.randperm(N,generator=g,device=device)[:NSUB]
    # OT: Sinkhorn on the subsample + nearest-sample extension to all cells
    t=tic()
    with torch.no_grad(): tt=sinkhorn_transport(sf[i_s],tf[i_t])
    nn=torch.cdist(sf,sf[i_s]).argmin(1); U=tt[nn]; t_ot=tic()-t
    # hard assignment: Hungarian on NEXACT samples + the same extension
    t=tic(); js,jt=i_s[:NEXACT],i_t[:NEXACT]; Cm=torch.cdist(sf[js],tf[jt]).pow(2).cpu().numpy(); r,c=linear_sum_assignment(Cm)
    th=torch.empty(NEXACT,C,device=device); th[torch.as_tensor(r,device=device)]=tf[jt][torch.as_tensor(c,device=device)]
    nnh=torch.cdist(sf,sf[js]).argmin(1); Uh=th[nnh]; t_hard=tic()-t
    # decode (one η)
    it=(0.5*sf+0.5*U).reshape(H,W,C).permute(2,0,1).unsqueeze(0); sm={s:(it if s==L else F.interpolate(it,size=s,mode='bilinear',align_corners=False)) for s in set(FMS)}
    t=tic(); out=decode_with_style_maps(ba,sm); t_dec=tic()-t
    # pixel-linear blend of style maps (its "method" step; then the same decode)
    t=tic(); smp={s:0.5*sa[s]+0.5*sb[s] for s in set(FMS)}; t_pix=tic()-t
    return dict(k=k,enc=t_enc,ot=t_ot,hard=t_hard,dec=t_dec,pix=t_pix)

for k in KS120[:3]: stages(k)                                            # warm-up (cuDNN autotune, allocator)
rows=[stages(k) for k in KS120]; T=pd.DataFrame(rows)
T['ot_midpoint']=T.enc+T.ot+T.dec; T['ot_interp']=T.ot+T.dec; T['ot_path11']=T.enc+T.ot+11*T.dec
T['hard_midpoint']=T.enc+T.hard+T.dec; T['pix_midpoint']=T.enc+T.pix+T.dec; T['pix_path11']=T.enc+T.pix+11*T.dec
T.to_csv(f'{BASE}/timing_120.csv',index=False)
med=T.median(numeric_only=True); q1=T.quantile(0.25,numeric_only=True); q3=T.quantile(0.75,numeric_only=True)
print(f'GPU {torch.cuda.get_device_name(0)} | n={len(T)} | median [IQR], seconds')
for c_ in ['enc','ot','hard','dec','pix','ot_midpoint','ot_interp','ot_path11','hard_midpoint','pix_midpoint','pix_path11']: print(f'  {c_:14} {med[c_]:.3f}  [{q1[c_]:.3f}, {q3[c_]:.3f}]')
MAC={'EncodeSec':med.enc,'OTsec':med.ot_midpoint,'OTInterpSec':med.ot_interp,'PathSecOT':med.ot_path11,'HardSec':med.hard_midpoint,'LinearSec':med.pix_midpoint,'PathSecLinear':med.pix_path11}
print('\n% numbers.tex, timing block (120 pairs, this run):'); print(''.join(f'\\newcommand{{\\{n}}}{{{v:.2f}}}' for n,v in MAC.items()))
json.dump({n:float(v) for n,v in MAC.items()},open(f'{BASE}/timing_120_macros.json','w'),indent=1)


In [ ]:
# ============================================================
# 6. MERGE — one table, one protocol, and the macro lines for numbers.tex
# ============================================================
ours=pd.read_csv(f'{BASE}/timing_120.csv'); tmt=pd.read_csv(f'{BASE}/timing_120_tm.csv'); assert set(ours.k)==set(tmt.k)
rows=[('OT (ours)',ours.enc,ours.ot+ours.dec,ours.ot_midpoint,ours.ot_path11),('Hard assignment (ours)',ours.enc,ours.hard+ours.dec,ours.hard_midpoint,ours.enc+ours.hard+11*ours.dec),
      ('Pixel-linear',ours.enc,ours.pix+ours.dec,ours.pix_midpoint,ours.pix_path11),('Texture Mixer',tmt.enc,tmt.gen,tmt.midpoint,tmt.path11)]
tab=pd.DataFrame([dict(method=n,encode=e.median(),per_eta=p.median(),midpoint=m.median(),midpoint_IQR=f'[{m.quantile(.25):.3f}, {m.quantile(.75):.3f}]',path11=pth.median()) for n,e,p,m,pth in rows])
print('median seconds on one A100, n=120 pairs, same protocol\n'); print(tab.round(3).to_string(index=False))
r=tab.set_index('method'); print(f"\nper-midpoint: TM / OT = {r.loc['Texture Mixer','midpoint']/r.loc['OT (ours)','midpoint']:.2f}x | per-eta step: TM / OT = {r.loc['Texture Mixer','per_eta']/r.loc['OT (ours)','per_eta']:.2f}x | 11-point path: TM / OT = {r.loc['Texture Mixer','path11']/r.loc['OT (ours)','path11']:.2f}x")
print('\n% numbers.tex timing block, 120 pairs:')
print(f"\\newcommand{{\\EncodeSec}}{{{r.loc['OT (ours)','encode']:.2f}}}\\newcommand{{\\OTsec}}{{{r.loc['OT (ours)','midpoint']:.2f}}}\\newcommand{{\\OTInterpSec}}{{{r.loc['OT (ours)','per_eta']:.2f}}}\\newcommand{{\\PathSecOT}}{{{r.loc['OT (ours)','path11']:.2f}}}")
print(f"\\newcommand{{\\HardSec}}{{{r.loc['Hard assignment (ours)','midpoint']:.2f}}}\\newcommand{{\\LinearSec}}{{{r.loc['Pixel-linear','midpoint']:.2f}}}\\newcommand{{\\PathSecLinear}}{{{r.loc['Pixel-linear','path11']:.2f}}}")
print(f"\\newcommand{{\\TMsec}}{{{r.loc['Texture Mixer','midpoint']:.2f}}}\\newcommand{{\\TMInterpSec}}{{{r.loc['Texture Mixer','per_eta']:.3f}}}\\newcommand{{\\PathSecTM}}{{{r.loc['Texture Mixer','path11']:.2f}}}")
print('\n(replaces both \\TMsec=0.375 and \\TMsecScale=0.023; delete \\TMsecScale)')


In [ ]:
# ============================================================
# PHASE 2, CELL 5b — OUR SIDE at 128 px (the resolution Texture Mixer runs at; the A/B files are natively 128)
# ============================================================
R=128; FMS128=[round(R*r) for r in model.generator.feature_map_size_ratios]
def stages128(k, seed=0):
    A,B=pair_imgs(k); ia,ib=cnt_load(A,R),cnt_load(B,R)
    t=tic()
    with torch.no_grad():
        ba=model.encode(ia); La=model.splat(ba,img_size=R)['layer_level_layouts'][-1][0]; bfa,sia=La['blob_features'].clone(),La['score_img'].clone()
        bb=model.encode(ib); Lb=model.splat(bb,img_size=R)['layer_level_layouts'][-1][0]; bfb,sib=Lb['blob_features'].clone(),Lb['score_img'].clone()
    t_enc=tic()-t
    sa=build_style_maps(bfa,sia,FMS128); sb=build_style_maps(bfb,sib,FMS128); L=max(FMS128); src,tgt=sa[L][0],sb[L][0]; C,H,W=src.shape
    sf=src.permute(1,2,0).reshape(-1,C); tf=tgt.permute(1,2,0).reshape(-1,C); N=sf.shape[0]
    g=torch.Generator(device=device).manual_seed(seed); i_s=torch.randperm(N,generator=g,device=device)[:NSUB]; i_t=torch.randperm(N,generator=g,device=device)[:NSUB]
    t=tic()
    with torch.no_grad(): tt=sinkhorn_transport(sf[i_s],tf[i_t])
    nn=torch.cdist(sf,sf[i_s]).argmin(1); U=tt[nn]; t_ot=tic()-t
    it=(0.5*sf+0.5*U).reshape(H,W,C).permute(2,0,1).unsqueeze(0); sm={s:(it if s==L else F.interpolate(it,size=s,mode='bilinear',align_corners=False)) for s in set(FMS128)}
    t=tic()
    with torch.no_grad():
        L_=model.splat(ba,img_size=R); si=L_['layer_level_layouts'][-1][0]['score_img']; sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(FMS128): s=F.interpolate(s,s.shape[-1]//2,mode='bilinear',align_corners=False); sp[s.shape[-1]]=s
        fi=sp[FMS128[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(FMS128)):
            x=model.generator.spade_blocks[i](x,sm[FMS128[i]])
            if i==len(FMS128)-1: break
            elif FMS128[i]!=FMS128[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode='bilinear',align_corners=False)
        out=torch.tanh(x.unsqueeze(1)).flatten(1,2)
    t_dec=tic()-t; assert out.shape[-1]==R
    return dict(k=k,enc=t_enc,ot=t_ot,dec=t_dec)
for k in KS120[:3]: stages128(k)
T8=pd.DataFrame([stages128(k) for k in KS120]); T8['midpoint']=T8.enc+T8.ot+T8.dec; T8['interp']=T8.ot+T8.dec; T8['path11']=T8.enc+T8.ot+11*T8.dec
T8.to_csv(f'{BASE}/timing_120_ours128.csv',index=False); m8=T8.median(numeric_only=True)
print('OT at 128 px, median s:',{c_:round(float(m8[c_]),3) for c_ in ['enc','ot','dec','midpoint','interp','path11']})
tmt=pd.read_csv(f'{BASE}/timing_120_tm.csv'); print(f"at matched 128 px: TM/OT per midpoint {tmt.midpoint.median()/m8.midpoint:.2f}x, per-eta {tmt.gen.median()/m8.interp:.2f}x")
print(f'\\newcommand{{\\OTsecLow}}{{{m8.midpoint:.2f}}}\\newcommand{{\\OTInterpSecLow}}{{{m8.interp:.3f}}}')